In [10]:
from typing import Annotated, Sequence, TypedDict

from UbuntuDrivers.detect import path_get_custom_supported_gpus
from jedi.plugins.stdlib import collections_namedtuple
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


from dotenv import load_dotenv
import os

from litellm import vector_store_index_registry

In [11]:
load_dotenv()

API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("BASE_URL")

In [12]:
llm = ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=0)

embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

pdf_path = "Stock_Market_Performance_2026.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")

pdf_loader = PyPDFLoader(pdf_path)

try:
    pages = pdf_loader.load()
    print(f"PDF has been loaded with {len(pages)} pages")
except Exception as e:
    print(f"Error loading PDF: {e}")
    raise

PDF has been loaded with 1 pages


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

pages_split = text_splitter.split_documents(pages)

persist_directory = r"./"
collection_name = "stock_market"

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

In [14]:
try:
    vector_store = Chroma.from_documents(
        documents=pages_split,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    print(f"Created ChromaDB vector store")
except Exception as e:
    print(f"Error setting up ChromaDB: {e}")
    raise

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5} # amount of chunks to return
)

@tool
def retriever_tool(query: str) -> str:
    """ This tool searches and returns the information from the Stock Market Performance document """
    docs = retriever.invoke(query)
    if not docs:
        return "I found no relevant information in the Stock Market Performance document."
    results = []
    for i, doc in enumerate(docs):
        results.append(f"Document {i+1}:\n{doc.page_content}")
    return "\n\n".join(results)

Created ChromaDB vector store


In [15]:
tools = [retriever_tool]

llm = llm.bind_tools(tools)

In [16]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def should_continue(state: AgentState):
    """Check if the last message contains tool calls."""
    result = state['messages'][-1]
    return hasattr(result, 'tool_calls') and len(result.tool_calls) > 0

system_prompt = """
You are an intelligent AI assistant who answers questions about Stock Market Performance based on the PDF document loaded into your knowledge base.
Use the retriever tool available to answer questions about the stock market performance data. You can make multiple calls if needed.
If you need to look up some information before asking a follow up question, you are allowed to do that!
Please always cite the specific parts of the documents you use in your answers.
"""

tools_dict = {our_tool.name: our_tool for our_tool in tools}

def call_llm(state: AgentState) -> AgentState:
    """ Function to call the LLM with the current state """
    messages = list(state['messages'])
    messages = [SystemMessage(content=system_prompt)] + messages
    message = llm.invoke(messages)
    return {'messages': [message]}

# Retriever Agent
def take_action(state: AgentState) -> AgentState:
    """ Execute tool calls from the LLM's response """
    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
        print(f"Calling Tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")

        if not t['name'] in tools_dict:
            print(f"\nTool: {t['name']} does not exist.")
            result = "Incorrect Tool Name, Please Retry and Select tool from List of Available tools."

        else:
            result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
            print(f"Result length: {len(str(result))}")

        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))

    print("Tools Execution Complete. Back to the model!")
    return {'messages': results}

In [17]:
LLM_NODE = "llm"
RETRIEVER_NODE = "retriever_agent"

graph = StateGraph(AgentState)
graph.add_node(LLM_NODE, call_llm)
graph.add_node(RETRIEVER_NODE, take_action)

graph.add_conditional_edges(
    LLM_NODE,
    should_continue,
    {True: RETRIEVER_NODE, False: END}
)
graph.add_edge(RETRIEVER_NODE, LLM_NODE)
graph.set_entry_point(LLM_NODE)

rag_agent = graph.compile()

In [18]:
def running_agent():
    print("\n=== RAG AGENT===")

    while True:
        user_input = input("\nWhat is your question: ")
        if user_input.lower() in ['exit', 'quit']:
            break

        messages = [HumanMessage(content=user_input)]

        result = rag_agent.invoke({"messages": messages})

        print("\n=== ANSWER ===")
        print(result['messages'][-1].content)

running_agent()


=== RAG AGENT===
Calling Tool: retriever_tool with query: IHSG 2027 prediction forecast
Result length: 3088
Tools Execution Complete. Back to the model!
Calling Tool: retriever_tool with query: 2027 IHSG prediction forecast outlook
Result length: 3088
Tools Execution Complete. Back to the model!

=== ANSWER ===
Based on the Stock Market Performance documents provided, I can analyze the current IHSG situation and trends, but there's **no specific 2027 prediction data** in the available documents. The documents only cover through 2026 and into Q4 2026 outlook.

## What the Documents Reveal About Current IHSG Status:

**2026 Performance:**
- IHSG peaked at **9,174.47** on January 19, 2026
- As of November 5, 2026: **5,030.26** (a 42.3% decline from peak)
- Worst month: October 2026 (-16.70% single-month crash)

**Critical Factors Impacting the Market:**
1. **Regulatory Shockwaves**: MSCI proposal to declassify Indonesia from Emerging Market to Standalone Market status
2. **Structural Saf

KeyboardInterrupt: Interrupted by user